In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q selenium
!pip install -q -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 499.2/499.2 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 10.3 MB/s eta 0:00:00


In [4]:
import sys
import os

project_root = '/content/drive/MyDrive/ai_enginner/job_search/AI/'
cache_dir = '/content/drive/MyDrive/ai_enginner/job_search/AI/cache/'
os.environ['HF_HOME'] = cache_dir
sys.path.append(project_root)
sys.path.append(cache_dir)

In [5]:
os.environ['HF_HOME']

'/content/drive/MyDrive/ai_enginner/job_search/AI/cache/'

In [22]:
from huggingface_hub import constants

print("HF_HOME:", os.environ.get("HF_HOME"))
print("현재 캐시 루트:", constants.HF_HUB_CACHE)
print("현재 캐시 루트:", constants.HF_HOME)

HF_HOME: /content/drive/MyDrive/ai_enginner/job_search/AI/cache/
현재 캐시 루트: /root/.cache/huggingface/hub


In [6]:
from transformers import BitsAndBytesConfig, AutoModelForCausalLM, AutoTokenizer
import torch

from src.utils import dict_to_str

In [8]:
device = "cuda"

# 모델, 토크나이저 로드
model_name = "skt/A.X-4.0-Light"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_storage=torch.bfloat16,
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [9]:
user_prompt = "취업 공고 정보를 분석하고, 취업 공고 정보를 요약해줘."
documents = [{'회사명': '', '채용제목': '[시드소프트]SI개발&웹개발&백엔드/서버개발 외 정규직 채용', '직무분야': '백엔드/서버개발,웹개발,SI개발,소프트웨어개발수정일 25/08/31', '근무지': '서울구로구', '경력': '신입', '학력': '학력무관', '마감일': '~ 09/08(월)', '지원링크': 'https://www.saramin.co.kr/zf_user/jobs/relay/view?view_type=search&rec_idx=51266037&location=ts&searchType=search&paid_fl=n&search_uuid=3514c091-8be8-46e3-8fdf-d9be17fb93f0'}, {'회사명': '', '채용제목': '영어회화 가능자', '직무분야': '백엔드/서버개발,기술지원,데이터엔지니어,IT컨설팅,SE(시스템엔지니어)외수정일 25/08/29', '근무지': '서울양천구', '경력': '신입', '학력': '학력무관', '마감일': '내일마감', '지원링크': 'https://www.saramin.co.kr/zf_user/jobs/relay/view?view_type=search&rec_idx=51688737&location=ts&searchType=search&paid_fl=n&search_uuid=3514c091-8be8-46e3-8fdf-d9be17fb93f0'}]

# docs 문자열로 변환
str_documents = ''

for i, doc_obj in enumerate(documents):
    str_documents += f'[{i}] \n '
    for key, value in doc_obj.items():
        str_documents += f'{key}: {value} | '
    str_documents += '\n'

print(str_documents)

[0] 
 회사명:  | 채용제목: [시드소프트]SI개발&웹개발&백엔드/서버개발 외 정규직 채용 | 직무분야: 백엔드/서버개발,웹개발,SI개발,소프트웨어개발수정일 25/08/31 | 근무지: 서울구로구 | 경력: 신입 | 학력: 학력무관 | 마감일: ~ 09/08(월) | 지원링크: https://www.saramin.co.kr/zf_user/jobs/relay/view?view_type=search&rec_idx=51266037&location=ts&searchType=search&paid_fl=n&search_uuid=3514c091-8be8-46e3-8fdf-d9be17fb93f0 | 
[1] 
 회사명:  | 채용제목: 영어회화 가능자 | 직무분야: 백엔드/서버개발,기술지원,데이터엔지니어,IT컨설팅,SE(시스템엔지니어)외수정일 25/08/29 | 근무지: 서울양천구 | 경력: 신입 | 학력: 학력무관 | 마감일: 내일마감 | 지원링크: https://www.saramin.co.kr/zf_user/jobs/relay/view?view_type=search&rec_idx=51688737&location=ts&searchType=search&paid_fl=n&search_uuid=3514c091-8be8-46e3-8fdf-d9be17fb93f0 | 



In [10]:
# instruct와 user 메시지 생성
chat = [
    {"role": "system", "content": "너는 취업 공고 정보 분석 전문가야. 취업 공고 정보를 분석하고, 취업 공고 정보를 요약하는 것이 너의 일이야.\n제공된 취업 공고 정보:\n" + str_documents},
    {"role": "user", "content": user_prompt}
]
print(chat)

[{'role': 'system', 'content': '너는 취업 공고 정보 분석 전문가야. 취업 공고 정보를 분석하고, 취업 공고 정보를 요약하는 것이 너의 일이야.\n제공된 취업 공고 정보:\n[0] \n 회사명:  | 채용제목: [시드소프트]SI개발&웹개발&백엔드/서버개발 외 정규직 채용 | 직무분야: 백엔드/서버개발,웹개발,SI개발,소프트웨어개발수정일 25/08/31 | 근무지: 서울구로구 | 경력: 신입 | 학력: 학력무관 | 마감일: ~ 09/08(월) | 지원링크: https://www.saramin.co.kr/zf_user/jobs/relay/view?view_type=search&rec_idx=51266037&location=ts&searchType=search&paid_fl=n&search_uuid=3514c091-8be8-46e3-8fdf-d9be17fb93f0 | \n[1] \n 회사명:  | 채용제목: 영어회화 가능자 | 직무분야: 백엔드/서버개발,기술지원,데이터엔지니어,IT컨설팅,SE(시스템엔지니어)외수정일 25/08/29 | 근무지: 서울양천구 | 경력: 신입 | 학력: 학력무관 | 마감일: 내일마감 | 지원링크: https://www.saramin.co.kr/zf_user/jobs/relay/view?view_type=search&rec_idx=51688737&location=ts&searchType=search&paid_fl=n&search_uuid=3514c091-8be8-46e3-8fdf-d9be17fb93f0 | \n'}, {'role': 'user', 'content': '취업 공고 정보를 분석하고, 취업 공고 정보를 요약해줘.'}]


In [11]:
# 메시지 토큰화
inputs = tokenizer.apply_chat_template(
            chat,
            add_generation_prompt=True,
            skip_reasoning=True,
            return_dict=True,
            return_tensors="pt"
        )
inputs = inputs.to(device)

In [ ]:
output_ids = model.generate(
    **inputs,
    max_length=1024,
    do_sample=True,
    stop_strings=["<|endofturn|>", "<|stop|>"],
    temperature=0.5,
    top_p=0.6,
    repetition_penalty=1.05,
    tokenizer=tokenizer
)

Both `max_new_tokens` (=16384) and `max_length`(=1024) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
len_input_prompt = len(inputs[0])
response = tokenizer.decode(output_ids[0][len_input_prompt:], skip_special_tokens=True)
print(response)